# Chavruta.AI — LoRA משגר (Colab / Kaggle / Lightning)

מחברת **משגר דקה** בלבד — כל לוגיקת האימון ב-`train_lora.py` (מקור אמת יחיד).

⚠️ **חובה:** `Runtime → Change runtime type → **GPU (T4)**` — לא TPU ולא CPU.
ה-stack (Unsloth + bitsandbytes) הוא CUDA-only ולא ירוץ על TPU.

💾 **Checkpoints נשמרים מקומית** ב-`/content/outputs/` (לא Drive). אחסון מקומי
ב-Colab זמני — התא האחרון מוריד את ה-adapter למחשב שלך בסוף.

📦 **דאטה מ-Hugging Face Hub** (repo פרטי) — קובץ האימון (162MB) גדול מדי ל-GitHub.

### 1. ודא שיש GPU

In [ ]:
!nvidia-smi

### 2. התקנה (פעם אחת לסשן)

In [ ]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes datasets huggingface_hub

### 3. הבא את הקוד + הדאטה מ-Hugging Face Hub

ה-repo **פרטי** — הרץ `login()` והדבק טוקן (Read מספיק) לפני המשיכה.

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download, login
login()  # repo פרטי — הדבק טוקן (Read מספיק). אם תהפוך לציבורי, אפשר למחוק שורה זו.

HF_DATASET = "Yehuda-Rubin/chavruta-torah-mixed"

os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/scripts", exist_ok=True)
os.chdir("/content")

downloads = {
    "torah_mixed_train.jsonl": "data/processed/torah_mixed_train.jsonl",
    "torah_mixed_val.jsonl":   "data/processed/torah_mixed_val.jsonl",
    "train_lora.py":           "scripts/train_lora.py",
}
for remote, local in downloads.items():
    p = hf_hub_download(repo_id=HF_DATASET, filename=remote, repo_type="dataset", local_dir="/content/_hf")
    shutil.copy(p, f"/content/{local}")
    print(f"✅ {os.path.getsize(f'/content/{local}')/1e6:7.2f} MB  {local}")

### 4. אימון — checkpoint מקומי ב-`/content/outputs`

שנה פרמטרים דרך ה-flags. אם OOM: הוסף `--max_seq 1024` או `--bs 1 --grad_accum 8`.

In [ ]:
!python scripts/train_lora.py \
  --model Qwen/Qwen3.5-4B \
  --epochs 1 \
  --max_seq 2048 \
  --bs 2 --grad_accum 4 \
  --out /content/outputs/chavruta-qwen35-4b-lora

### 5. הורד את ה-adapter (אחסון מקומי זמני — הורד לפני ניתוק!)

In [ ]:
import shutil
adapter_dir = "/content/outputs/chavruta-qwen35-4b-lora"
zip_path = shutil.make_archive("/content/chavruta_adapter", "zip", adapter_dir)
print("📦", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("לא Colab או אין דפדפן — הורד ידנית מהסרגל:", zip_path)

---
### הערות
- **דאטה ב-HF, קוד ב-GitHub:** קובץ האימון 162MB > מגבלת 100MB של GitHub, לכן הופרד ל-HF.
  כשתגדיל את הדאטה (מפרשים / שאר התנ"ך) — הרץ שוב `upload_dataset_hf.py` והמחברת תמשוך מעודכן.
- **checkpoint מקומי** לפי בקשתך (זמני) — תא 5 מוריד את ה-adapter. לריצות ארוכות שקול `--out /content/drive/...` אחרי mount.
- **Qwen3.5 חדש:** אם Unsloth לא מזהה — עדכן לאחרון. אם תבנית ה-chat שונה מ-ChatML — תקן את המסמנים ב-`train_on_responses_only` בתוך `train_lora.py`.
- **Kaggle/Lightning:** אותה מחברת; רק תא 5 (`files.download`) הוא Colab-ספציפי.